# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
%matplotlib inline
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]
tree_preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), feature_columns)])
scaled_preprocessor = ColumnTransformer([("numeric", Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())]), feature_columns)])

**Respuesta:** El scaler debe calcular media y desviación exclusivamente con el subconjunto de entrenamiento de cada fold. Ajustarlo antes de CV filtraría información de los folds de validación al entrenamiento y sesgaría la estimación. El Pipeline también asegura que imputación, escalado e inferencia sean consistentes.

## 3. Modelos candidatos

In [5]:
models = {
    "DecisionTree": Pipeline([("preprocess", clone(tree_preprocessor)),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE))]),
    "LogisticRegression": Pipeline([("preprocess", clone(scaled_preprocessor)),
        ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))]),
    "KNN": Pipeline([("preprocess", clone(scaled_preprocessor)),
        ("model", KNeighborsClassifier())]),
}

## 4. Comparación inicial con validación cruzada

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, pipeline in models.items():
    scores = cross_validate(pipeline, X_train, y_train, cv=cv,
                            scoring="accuracy", return_train_score=True)
    rows.append({"model": name, "train_accuracy": scores["train_score"].mean(),
                 "validation_accuracy": scores["test_score"].mean(),
                 "validation_std": scores["test_score"].std()})
initial_comparison = pd.DataFrame(rows)
display(initial_comparison)

,model,train_accuracy,validation_accuracy,validation_std
0,DecisionTree,1.0000,0.902217,0.058617
1,LogisticRegression,1.0000,0.979064,0.027713
2,KNN,0.9771,0.950739,0.035274


## 5. Optimización de hiperparámetros

In [7]:
param_grids = {
    "DecisionTree": {"model__max_depth": [2, 3, 5, None], "model__min_samples_leaf": [1, 3, 5]},
    "LogisticRegression": {"model__C": [0.01, 0.1, 1, 10]},
    "KNN": {"model__n_neighbors": [3, 5, 7, 11], "model__weights": ["uniform", "distance"]},
}
searches = {}
rows = []
for name, pipeline in models.items():
    search = GridSearchCV(pipeline, param_grids[name], scoring="accuracy", cv=cv,
                          return_train_score=True, n_jobs=1)
    search.fit(X_train, y_train)
    searches[name] = search
    k = search.best_index_
    rows.append({"model": name, "cv_accuracy": search.best_score_,
        "cv_std": search.cv_results_["std_test_score"][k],
        "train_accuracy": search.cv_results_["mean_train_score"][k],
        "params": search.best_params_})
tuned_comparison = pd.DataFrame(rows).sort_values("cv_accuracy", ascending=False)
display(tuned_comparison)

,model,cv_accuracy,cv_std,train_accuracy,params
1,LogisticRegression,0.979064,0.027713,1.000000,{'model__C': 1}
2,KNN,0.957882,0.040520,0.964773,"{'model__n_neighbors': 7, 'model__weights': 'u..."
0,DecisionTree,0.902217,0.058617,1.000000,"{'model__max_depth': 5, 'model__min_samples_le..."


## 6. Selección y persistencia del campeón

In [8]:
champion_name = max(searches, key=lambda name: searches[name].best_score_)
champion = searches[champion_name].best_estimator_
search = searches[champion_name]
k = search.best_index_
metadata = {"champion": champion_name, "cv_accuracy": float(search.best_score_),
    "cv_std": float(search.cv_results_["std_test_score"][k]),
    "train_accuracy": float(search.cv_results_["mean_train_score"][k]),
    "best_params": search.best_params_, "random_state": RANDOM_STATE, "n_splits": 5,
    "selection_rule": "Maximum mean CV accuracy; deterministic model order breaks ties"}
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")
(ARTIFACTS_DIR / "training_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
pd.concat([pd.DataFrame(s.cv_results_).assign(model=name) for name, s in searches.items()],
          ignore_index=True).to_csv(REPORTS_DIR / "cv_results.csv", index=False)
print("Campeón:", champion_name)
print(f"CV: {metadata['cv_accuracy']:.4f} ± {metadata['cv_std']:.4f}")
print(f"Brecha train-CV: {metadata['train_accuracy'] - metadata['cv_accuracy']:.4f}")

Campeón: LogisticRegression
CV: 0.9791 ± 0.0277
Brecha train-CV: 0.0209


In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:** Se aplica la regla fijada antes de abrir test: mayor accuracy promedio en los mismos cinco folds estratificados. La tabla muestra dispersión y brecha entrenamiento-validación, evitando decidir por accuracy de entrenamiento. LogisticRegression ofrece coeficientes interpretables y regularización; el árbol es fácil de explicar pero puede sobreajustar; KNN es sencillo, aunque depende del escalado y conserva las muestras. Las diferencias pequeñas entre candidatos no demuestran superioridad estadística: esta CV también se utilizó para optimizar hiperparámetros y podría ser optimista. Test se mantiene reservado para la siguiente etapa.

## Checklist

- [x] Preprocesamiento dentro de pipelines.
- [x] Tres familias comparadas y optimizadas con CV.
- [x] Campeón seleccionado sin cargar test.
- [x] Pipeline, resultados y metadatos guardados.

### Resultado de esta ejecución

El campeón fue **LogisticRegression**, con C=1, accuracy CV **0,9791 ± 0,0277** y accuracy de entrenamiento promedio 1,0. La brecha de 2,09 puntos sugiere cautela, aunque es pequeña respecto a la dispersión entre folds. Se conserva por la regla predefinida de máximo promedio y por su baja complejidad relativa; la elección no utilizó test.
